In [1]:
## 1. Import Libraries and Set Reproducibility

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

# Fixed random seed
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

print("Random seed set to:", RANDOM_SEED)

Random seed set to: 42


In [2]:
## 2. Mount Google Drive and Load Final Manifests

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Master_datascience_thesis"

preprocessing_dir = os.path.join(PROJECT_ROOT, "outputs", "preprocessing")
model_output_dir = os.path.join(PROJECT_ROOT, "outputs", "deep_learning")

train_df = pd.read_csv(os.path.join(preprocessing_dir, "final_train_metadata.csv"))

validation_df = pd.read_csv(os.path.join(preprocessing_dir, "final_validation_metadata.csv"))

print("Training images:", len(train_df))
print("Validation images:", len(validation_df))

Mounted at /content/drive
Training images: 6480
Validation images: 1794


In [3]:
## 3. Load Existing Trained Models

resnet50_model = tf.keras.models.load_model(os.path.join(model_output_dir, "best_resnet50_model.keras"))

mobilenetv2_model = tf.keras.models.load_model(os.path.join(model_output_dir, "best_mobilenetv2_model.keras"))

print("ResNet50 loaded successfully")
print("MobileNetV2 loaded successfully")

ResNet50 loaded successfully
MobileNetV2 loaded successfully


In [4]:
## 4. Inspect Model Layers

print("ResNet50 layers:")
for i, layer in enumerate(resnet50_model.layers):
    print(i, layer.name, type(layer).__name__)

print("\nMobileNetV2 layers:")
for i, layer in enumerate(mobilenetv2_model.layers):
    print(i, layer.name, type(layer).__name__)

ResNet50 layers:
0 input_layer_1 InputLayer
1 sequential Sequential
2 resnet50 Functional
3 global_average_pooling2d GlobalAveragePooling2D
4 dense Dense

MobileNetV2 layers:
0 input_layer_4 InputLayer
1 sequential Sequential
2 mobilenetv2_1.00_224 Functional
3 global_average_pooling2d_1 GlobalAveragePooling2D
4 dense_1 Dense


In [5]:
## 5. Inspect Upper Backbone Layers

resnet50_base = resnet50_model.get_layer("resnet50")
mobilenetv2_base = mobilenetv2_model.get_layer("mobilenetv2_1.00_224")

print("ResNet50 backbone layers:", len(resnet50_base.layers))
print("\nLast 30 ResNet50 layers:")
for i, layer in enumerate(resnet50_base.layers[-30:], start=len(resnet50_base.layers)-30):
    print(i, layer.name, type(layer).__name__)

print("\nMobileNetV2 backbone layers:", len(mobilenetv2_base.layers))
print("\nLast 30 MobileNetV2 layers:")
for i, layer in enumerate(mobilenetv2_base.layers[-30:], start=len(mobilenetv2_base.layers)-30):
    print(i, layer.name, type(layer).__name__)

ResNet50 backbone layers: 175

Last 30 ResNet50 layers:
145 conv5_block1_1_relu Activation
146 conv5_block1_2_conv Conv2D
147 conv5_block1_2_bn BatchNormalization
148 conv5_block1_2_relu Activation
149 conv5_block1_0_conv Conv2D
150 conv5_block1_3_conv Conv2D
151 conv5_block1_0_bn BatchNormalization
152 conv5_block1_3_bn BatchNormalization
153 conv5_block1_add Add
154 conv5_block1_out Activation
155 conv5_block2_1_conv Conv2D
156 conv5_block2_1_bn BatchNormalization
157 conv5_block2_1_relu Activation
158 conv5_block2_2_conv Conv2D
159 conv5_block2_2_bn BatchNormalization
160 conv5_block2_2_relu Activation
161 conv5_block2_3_conv Conv2D
162 conv5_block2_3_bn BatchNormalization
163 conv5_block2_add Add
164 conv5_block2_out Activation
165 conv5_block3_1_conv Conv2D
166 conv5_block3_1_bn BatchNormalization
167 conv5_block3_1_relu Activation
168 conv5_block3_2_conv Conv2D
169 conv5_block3_2_bn BatchNormalization
170 conv5_block3_2_relu Activation
171 conv5_block3_3_conv Conv2D
172 conv5_blo

In [6]:
## 6. Configure Fine-Tuning Layers

# -------------------------
# ResNet50
# -------------------------
resnet50_base.trainable = True

# Freeze everything first
for layer in resnet50_base.layers:
    layer.trainable = False

# Unfreeze final residual block only
unfreeze = False

for layer in resnet50_base.layers:
    if layer.name == "conv5_block3_1_conv":
        unfreeze = True

    if unfreeze and not isinstance(
        layer, tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True


# -------------------------
# MobileNetV2
# -------------------------
mobilenetv2_base.trainable = True

# Freeze everything first
for layer in mobilenetv2_base.layers:
    layer.trainable = False

# Unfreeze final inverted-residual block + final convolution
unfreeze = False

for layer in mobilenetv2_base.layers:
    if layer.name == "block_16_expand":
        unfreeze = True

    if unfreeze and not isinstance(
        layer, tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True


print("ResNet50 trainable backbone layers:")
for layer in resnet50_base.layers:
    if layer.trainable:
        print(layer.name)

print("\nMobileNetV2 trainable backbone layers:")
for layer in mobilenetv2_base.layers:
    if layer.trainable:
        print(layer.name)

ResNet50 trainable backbone layers:
conv5_block3_1_conv
conv5_block3_1_relu
conv5_block3_2_conv
conv5_block3_2_relu
conv5_block3_3_conv
conv5_block3_add
conv5_block3_out

MobileNetV2 trainable backbone layers:
block_16_expand
block_16_expand_relu
block_16_depthwise
block_16_depthwise_relu
block_16_project
Conv_1
out_relu


In [7]:
## 7. Recompile Models for Fine-Tuning

fine_tune_lr = 1e-5

resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auroc")
    ]
)

mobilenetv2_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auroc")
    ]
)

print("Models recompiled with learning rate:", fine_tune_lr)

Models recompiled with learning rate: 1e-05


In [8]:
## 8. Check Metadata Columns

print("Training columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(validation_df.columns.tolist())

print("\nFirst training row:")
display(train_df.head(1))

Training columns:
['path', 'split', 'original_class', 'binary_label', 'channel_max_diff', 'channels_identical', 'mean_intensity', 'std_intensity', 'pixel_hash', 'border_center_diff', 'corner_variation', 'label']

Validation columns:
['path', 'split', 'original_class', 'binary_label', 'channel_max_diff', 'channels_identical', 'mean_intensity', 'std_intensity', 'pixel_hash', 'border_center_diff', 'corner_variation', 'label']

First training row:


,path,split,original_class,binary_label,channel_max_diff,channels_identical,mean_intensity,std_intensity,pixel_hash,border_center_diff,corner_variation,label
0,sick/s0230.png,train,sick,Non-TB,0,True,132.578995,57.886063,000a236dc74b5d84150c7f05d8c9d70170d27f8bba1345...,33.350174,55.760803,0


In [3]:
## 9. Set Image Path and Image Loader

import zipfile
from PIL import Image

zip_path = (
    "/content/drive/MyDrive/"
    "Master_datascience_thesis/data/TBX11K.zip"
)

IMAGE_SIZE = (224, 224)

def load_image(zip_file, image_path):
    full_path = "TBX11K/imgs/" + image_path

    with zip_file.open(full_path) as file:
        image = Image.open(file).convert("RGB")
        image = image.resize(
            IMAGE_SIZE,
            Image.Resampling.BILINEAR
        )

    return np.array(image)

print("TBX11K ZIP found:", os.path.exists(zip_path))
print("Image size:", IMAGE_SIZE)

TBX11K ZIP found: True
Image size: (224, 224)


In [10]:
## 10. Calculate Class Weights

from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print("Class weights:", class_weights)

Class weights: {0: np.float64(0.5510204081632653), 1: np.float64(5.4)}


In [4]:
## 11. Create Training and Validation Data Generators

BATCH_SIZE = 32

class ImageSequence(tf.keras.utils.Sequence):

    def __init__(self, dataframe, batch_size, shuffle=False):
        super().__init__()
        self.dataframe = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        batch_indices = self.indices[start:end]
        batch_df = self.dataframe.iloc[batch_indices]

        images = []

        with zipfile.ZipFile(zip_path, "r") as zip_file:
            for image_path in batch_df["path"]:
                image = load_image(zip_file, image_path)
                images.append(image)

        images = np.array(images, dtype=np.float32)
        labels = batch_df["label"].to_numpy(dtype=np.float32)

        return images, labels

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


train_data = ImageSequence(
    train_df,
    BATCH_SIZE,
    shuffle=True
)

validation_data = ImageSequence(
    validation_df,
    BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_data))
print("Validation batches:", len(validation_data))

Training batches: 203
Validation batches: 57


In [12]:
## 12. Fine-Tuning Callbacks

resnet50_finetuned_path = os.path.join(
    model_output_dir,
    "best_resnet50_finetuned.keras"
)

mobilenetv2_finetuned_path = os.path.join(
    model_output_dir,
    "best_mobilenetv2_finetuned.keras"
)

resnet50_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        resnet50_finetuned_path,
        monitor="val_auroc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auroc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

mobilenetv2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        mobilenetv2_finetuned_path,
        monitor="val_auroc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auroc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

print("Fine-tuning callbacks ready")

Fine-tuning callbacks ready


In [13]:
print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [14]:
## 14. Fine-Tune ResNet50

resnet50_history = resnet50_model.fit(
    train_data,
    validation_data=validation_data,
    epochs=10,
    class_weight=class_weights,
    callbacks=resnet50_callbacks,
    verbose=1
)

Epoch 1/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9311 - auroc: 0.9740 - loss: 0.2171 - precision: 0.6007 - recall: 0.9231
Epoch 1: val_auroc improved from None to 0.97816, saving model to /content/drive/MyDrive/Master_datascience_thesis/outputs/deep_learning/best_resnet50_finetuned.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Master_datascience_thesis/outputs/deep_learning/best_resnet50_finetuned.keras
203/203 ━━━━━━━━━━━━━━━━━━━━ 316s 1s/step - accuracy: 0.9364 - auroc: 0.9785 - loss: 0.1939 - precision: 0.6022 - recall: 0.9233 - val_accuracy: 0.9415 - val_auroc: 0.9782 - val_loss: 0.1551 - val_precision: 0.6955 - val_recall: 0.8450
Epoch 2/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 584ms/step - accuracy: 0.9273 - auroc: 0.9792 - loss: 0.1979 - precision: 0.5793 - recall: 0.9152
Epoch 2: val_auroc improved from 0.97816 to 0.97845, saving model to /content/drive/MyDrive/Master_datascience_thesis/outputs/deep_learning/best_resnet50_finetuned.keras

Epoch 2: fi

In [15]:
## 15. Fine-Tune MobileNetV2

mobilenetv2_history = mobilenetv2_model.fit(
    train_data,
    validation_data=validation_data,
    epochs=10,
    class_weight=class_weights,
    callbacks=mobilenetv2_callbacks,
    verbose=1
)

Epoch 1/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 592ms/step - accuracy: 0.9270 - auroc: 0.9737 - loss: 0.1955 - precision: 0.5674 - recall: 0.9244
Epoch 1: val_auroc improved from None to 0.96607, saving model to /content/drive/MyDrive/Master_datascience_thesis/outputs/deep_learning/best_mobilenetv2_finetuned.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Master_datascience_thesis/outputs/deep_learning/best_mobilenetv2_finetuned.keras
203/203 ━━━━━━━━━━━━━━━━━━━━ 163s 767ms/step - accuracy: 0.9211 - auroc: 0.9747 - loss: 0.2061 - precision: 0.5440 - recall: 0.9167 - val_accuracy: 0.9476 - val_auroc: 0.9661 - val_loss: 0.1452 - val_precision: 0.7190 - val_recall: 0.8700
Epoch 2/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 0s 569ms/step - accuracy: 0.9287 - auroc: 0.9781 - loss: 0.1930 - precision: 0.5816 - recall: 0.9233
Epoch 2: val_auroc did not improve from 0.96607
203/203 ━━━━━━━━━━━━━━━━━━━━ 148s 731ms/step - accuracy: 0.9287 - auroc: 0.9801 - loss: 0.1825 - precision: 0.5708 - recal

In [5]:
## 16. Load Fine-Tuned Models

resnet50_finetuned = tf.keras.models.load_model(
    os.path.join(
        model_output_dir,
        "best_resnet50_finetuned.keras"
    )
)

mobilenetv2_finetuned = tf.keras.models.load_model(
    os.path.join(
        model_output_dir,
        "best_mobilenetv2_finetuned.keras"
    )
)

print("Fine-tuned ResNet50 loaded")
print("Fine-tuned MobileNetV2 loaded")

Fine-tuned ResNet50 loaded
Fine-tuned MobileNetV2 loaded


In [6]:
## 17. Get Validation Predictions

resnet50_finetuned_scores = resnet50_finetuned.predict(
    validation_data,
    verbose=1
).ravel()

mobilenetv2_finetuned_scores = mobilenetv2_finetuned.predict(
    validation_data,
    verbose=1
).ravel()

y_true = validation_df["label"].to_numpy()

print("ResNet50 predictions:", len(resnet50_finetuned_scores))
print("MobileNetV2 predictions:", len(mobilenetv2_finetuned_scores))
print("True labels:", len(y_true))

57/57 ━━━━━━━━━━━━━━━━━━━━ 417s 7s/step
57/57 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step
ResNet50 predictions: 1794
MobileNetV2 predictions: 1794
True labels: 1794


In [7]:
## 18. Calculate Fine-Tuned Validation Metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def calculate_metrics(y_true, scores):
    predictions = (scores >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions
    ).ravel()

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions),
        "Sensitivity": recall_score(y_true, predictions),
        "Specificity": tn / (tn + fp),
        "F1_score": f1_score(y_true, predictions),
        "AUROC": roc_auc_score(y_true, scores),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


resnet50_ft_metrics = calculate_metrics(
    y_true,
    resnet50_finetuned_scores
)

mobilenetv2_ft_metrics = calculate_metrics(
    y_true,
    mobilenetv2_finetuned_scores
)

print("Fine-tuned ResNet50:")
print(resnet50_ft_metrics)

print("\nFine-tuned MobileNetV2:")
print(mobilenetv2_ft_metrics)

Fine-tuned ResNet50:
{'Accuracy': 0.9470457079152731, 'Precision': 0.6895306859205776, 'Sensitivity': 0.955, 'Specificity': np.float64(0.946047678795483), 'F1_score': 0.80083857442348, 'AUROC': np.float64(0.9873117942283564), 'TN': np.int64(1508), 'FP': np.int64(86), 'FN': np.int64(9), 'TP': np.int64(191)}

Fine-tuned MobileNetV2:
{'Accuracy': 0.947603121516165, 'Precision': 0.71900826446281, 'Sensitivity': 0.87, 'Specificity': np.float64(0.9573400250941029), 'F1_score': 0.7873303167420814, 'AUROC': np.float64(0.9666624843161856), 'TN': np.int64(1526), 'FP': np.int64(68), 'FN': np.int64(26), 'TP': np.int64(174)}


In [8]:
## 19. Save Fine-Tuned Results and Scores

fine_tuned_results = pd.DataFrame([
    {"Model": "ResNet50", **resnet50_ft_metrics},
    {"Model": "MobileNetV2", **mobilenetv2_ft_metrics}
])

fine_tuned_results.to_csv(
    os.path.join(
        model_output_dir,
        "deep_learning_finetuned_validation_results.csv"
    ),
    index=False
)


fine_tuned_scores = validation_df[["path", "label"]].copy()

fine_tuned_scores["ResNet50_score"] = resnet50_finetuned_scores
fine_tuned_scores["MobileNetV2_score"] = mobilenetv2_finetuned_scores

fine_tuned_scores.to_csv(
    os.path.join(
        model_output_dir,
        "deep_learning_finetuned_validation_scores.csv"
    ),
    index=False
)

print("Fine-tuned results saved")
print("Fine-tuned validation scores saved")

Fine-tuned results saved
Fine-tuned validation scores saved
